# Maya4 Quick Start

This notebook is the shortest end-to-end example of the Maya4 online workflow.

What you will learn:
- how the package points to the public Hugging Face bucket `ESA-philab/Maya4`
- how `SampleFilter` narrows the remote catalog before sampling patches
- what a single `(x, y)` training batch looks like once it has been cached locally

The notebook keeps the workload intentionally small: one product, one sample, one batch.


## Step 1: Set Up the Environment

We start by locating the repository root, creating the local cache directory, and importing the public entry points used throughout the tutorial.

The cache layout for online mode is flat: `data/<product>.zarr`.


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from maya4 import DEFAULT_BUCKET_ID, SampleFilter, get_sar_dataloader

DATA_DIR = REPO_ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)

print('bucket_id:', DEFAULT_BUCKET_ID)
print('repo_root:', REPO_ROOT.name)
print('local_cache_dir:', DATA_DIR.relative_to(REPO_ROOT).as_posix())


bucket_id: ESA-philab/Maya4
repo_root: Maya4
local_cache_dir: data


## Step 2: Define a Small, Readable Filter

`SampleFilter` decides which remote products are eligible before patch sampling happens.

In this example we ask for:
- year `2025`
- `VV` polarization
- stripmap modes `S1` and `S4`

Keeping the filter explicit makes it easier to reason about why a product was selected.


In [2]:
filters = SampleFilter(
    years=[2025],
    polarizations=['vv'],
    stripmap_modes=[1, 4],
)

print('Configured filter:')
print('  years ->', filters.years)
print('  polarizations ->', filters.polarizations)
print('  stripmap_modes ->', filters.stripmap_modes)


Configured filter:
  years -> [2025]
  polarizations -> ['vv']
  stripmap_modes -> [1, 4]


## Step 3: Build the Dataloader

The dataloader is configured for a tiny online read:
- `online=True` means Maya4 will download only what it needs from the bucket
- `max_products=1` and `samples_per_prod=1` keep the example deterministic and fast
- `level_from='rcmc'` and `level_to='az'` make the batch contain one input tensor and one target tensor

This is the same API you would use in training code, just with much smaller limits.


In [3]:
loader = get_sar_dataloader(
    data_dir=str(DATA_DIR),
    bucket_id=DEFAULT_BUCKET_ID,
    filters=filters,
    level_from='rcmc',
    level_to='az',
    batch_size=1,
    patch_size=(128, 128),
    stride=(128, 128),
    buffer=(0, 0),
    num_workers=0,
    max_products=1,
    samples_per_prod=1,
    online=True,
    use_balanced_sampling=False,
    verbose=False,
)

print('Dataloader ready.')
print('  batch_size ->', loader.batch_size)
print('  dataset_size ->', len(loader.dataset))
print('  online_mode ->', loader.dataset.online)


Dataloader ready.
  batch_size -> 1
  dataset_size -> 1
  online_mode -> True


## Step 4: Read One Batch and Inspect It

At this point the first product has been resolved, cached locally, and converted into a training patch pair.

Things to notice in the output:
- `selected_files` tells you which remote product backed the batch
- `x_batch` and `y_batch` should have the same spatial shape in this example
- the dtype confirms the tensors were loaded successfully from Zarr


In [4]:
x_batch, y_batch = next(iter(loader))
selected_files = [Path(f).name for f in loader.dataset.get_files()[:1]]

print('selected_files:', selected_files)
print('x_batch shape:', tuple(x_batch.shape), 'dtype:', x_batch.dtype)
print('y_batch shape:', tuple(y_batch.shape), 'dtype:', y_batch.dtype)
print('x_batch mean(abs):', float(x_batch.abs().mean()))
print('y_batch mean(abs):', float(y_batch.abs().mean()))


selected_files: ['s1c-s1-raw-s-vv-20250328t052810-20250328t052835-001637-002a0d.zarr']
x_batch shape: (1, 128, 128, 4) dtype: torch.float64
y_batch shape: (1, 128, 128, 4) dtype: torch.float64
x_batch mean(abs): 1171.022624902179
y_batch mean(abs): 1069.737963989555


## Takeaways

You now have a minimal template for online Maya4 access.

From here you can scale up by changing only a few parameters:
- broaden `SampleFilter` to include more dates, modes, or polarizations
- increase `max_products` or `samples_per_prod` to grow the training set
- switch `level_from` and `level_to` if you want a different processing-stage pair
